In [3]:
import json
import csv
import re
import unicodedata
from pathlib import Path
from collections import defaultdict

IN_PATH = Path("data/financebench_merged.jsonl")
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [4]:
def clean_text(text: str) -> str:
    """Normalize whitespace/unicode without touching numbers or line items."""
    if not text:
        return text
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\n{3,}", "\n\n", text)          # collapse blank-line runs
    text = re.sub(r"[ \t]+\n", "\n", text)           # trailing spaces per line
    text = re.sub(r"[ \t]{2,}", " ", text)           # collapse multi-spaces
    return text.strip()

In [5]:
records = [json.loads(line) for line in IN_PATH.open(encoding="utf-8")]
print(f"Loaded {len(records)} records")
records[0]  # peek at one to confirm it loaded right

Loaded 150 records


{'financebench_id': 'financebench_id_03029',
 'company': '3M',
 'doc_name': '3M_2018_10K',
 'question_type': 'metrics-generated',
 'question_reasoning': 'Information extraction',
 'domain_question_num': None,
 'question': 'What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.',
 'answer': '$1577.00',
 'justification': 'The metric capital expenditures was directly extracted from the company 10K. The line item name, as seen in the 10K, was: Purchases of property, plant and equipment (PP&E).',
 'dataset_subset_label': 'OPEN_SOURCE',
 'evidence': [{'evidence_text': 'Table of Contents \n3M Company and Subsidiaries\nConsolidated Statement of Cash Flow s\nYears ended December 31\n \n(Millions)\n \n2018\n \n2017\n \n2016\n \nCash Flows from Operating Activities\n \n \n \n \n \n \n \nNet income including noncontrolling interest\n \n$\n5,363 \n$\n4,869 \n$\n5,058 \nAdjustments to reconci

In [6]:
pages = {}      # (doc_name, page_num) -> page record
doc_meta = {}   # doc_name -> filing-level metadata

for r in records:
    doc_meta[r["doc_name"]] = {
        "doc_name": r["doc_name"],
        "company": r["company"],
        "doc_type": r["doc_type"],
        "doc_period": r["doc_period"],
        "gics_sector": r["gics_sector"],
        "doc_link": r["doc_link"],
    }
    for ev in r["evidence"]:
        key = (ev["doc_name"], ev["evidence_page_num"])
        if key not in pages:
            pages[key] = {
                "page_id": f"{ev['doc_name']}_p{ev['evidence_page_num']}",
                "doc_name": ev["doc_name"],
                "company": r["company"],
                "doc_type": r["doc_type"],
                "doc_period": r["doc_period"],
                "gics_sector": r["gics_sector"],
                "page_num": ev["evidence_page_num"],
                "text": clean_text(ev["evidence_text_full_page"]),
                "doc_link": r["doc_link"],
            }

print(f"{len(pages)} unique chunks (pages) from {len(doc_meta)} filings")

168 unique chunks (pages) from 84 filings


In [7]:
corpus_path = OUT_DIR / "corpus_pages.jsonl"
with corpus_path.open("w", encoding="utf-8") as f:
    for key in sorted(pages, key=lambda k: (k[0], k[1])):
        f.write(json.dumps(pages[key], ensure_ascii=False) + "\n")
print(f"Wrote {corpus_path}")

Wrote data/processed/corpus_pages.jsonl


In [8]:
eval_path = OUT_DIR / "eval_questions.jsonl"
with eval_path.open("w", encoding="utf-8") as f:
    for r in records:
        gold_pages = [f"{ev['doc_name']}_p{ev['evidence_page_num']}" for ev in r["evidence"]]
        row = {
            "id": r["financebench_id"],
            "question": r["question"],
            "answer": (r.get("answer") or "").strip(),
            "justification": r.get("justification", ""),
            "question_type": r["question_type"],
            "question_reasoning": r.get("question_reasoning", ""),
            "company": r["company"],
            "doc_name": r["doc_name"],
            "gold_evidence_pages": gold_pages,
            "evidence_snippets": [clean_text(ev["evidence_text"]) for ev in r["evidence"]],
        }
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"Wrote {eval_path}")

Wrote data/processed/eval_questions.jsonl


In [9]:
manifest_path = OUT_DIR / "docs_manifest.csv"
pages_per_doc = defaultdict(int)
for key in pages:
    pages_per_doc[key[0]] += 1

with manifest_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "doc_name", "company", "doc_type", "doc_period",
        "gics_sector", "doc_link", "pages_in_corpus",
    ])
    writer.writeheader()
    for doc_name in sorted(doc_meta):
        writer.writerow({**doc_meta[doc_name], "pages_in_corpus": pages_per_doc[doc_name]})
print(f"Wrote {manifest_path}")

Wrote data/processed/docs_manifest.csv


In [10]:
corpus_ids = set(pages[k]["page_id"] for k in pages)
missing = 0
for r in records:
    for ev in r["evidence"]:
        pid = f"{ev['doc_name']}_p{ev['evidence_page_num']}"
        if pid not in corpus_ids:
            missing += 1
            print("MISSING:", r["financebench_id"], pid)
print(f"Gold evidence pages missing from corpus: {missing} (should be 0)")

Gold evidence pages missing from corpus: 0 (should be 0)
